<a href="https://colab.research.google.com/github/kaziibtisumwasit/Thesis_Parkinson-s-Disease-Detection-from-Voice-Recording/blob/main/Parkinson%E2%80%99s_Disease_Detection_from_Voice_Recordings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# import libraries

In [19]:
## import libraries
import os ## for folder & path
import glob # finds
import numpy as np # numerical calculations
import pandas as pd # from dataframe
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import zipfile ## for unziping files


In [5]:
!pip install librosa soundfile numpy pandas

# load raw files

In [9]:
## read data from drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
## import data this drive folder path
import os
os.listdir('/content/drive/MyDrive/Colab Notebooks/Thesis/Parkinson’s Disease Detection from Voice Recordings/raw_data/data')

['.DS_Store', 'Italian', 'english']

In [15]:
## store this raw data folder path into variable
DATA_ROOT = "/content/drive/MyDrive/Colab Notebooks/Thesis/Parkinson’s Disease Detection from Voice Recordings/raw_data/data"

In [17]:
## Test raw data root
print(os.listdir(DATA_ROOT))

['.DS_Store', 'Italian', 'english']


## Extract features

---



In [25]:
SR = 16000
N_MFCC = 13

def extract_features(filepath):
    y, _ = librosa.load(filepath, sr=SR, mono=True)  # keep silence (no trimming)

    if len(y) < int(SR * 0.3):
        return None

    feats = {}

    mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC)
    dmfcc = librosa.feature.delta(mfcc)

    rms = librosa.feature.rms(y=y).flatten()
    zcr = librosa.feature.zero_crossing_rate(y).flatten()
    centroid = librosa.feature.spectral_centroid(y=y, sr=SR).flatten()
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=SR).flatten()

    def add_stats(prefix, arr):
        feats[f"{prefix}_mean"] = float(np.mean(arr))
        feats[f"{prefix}_std"]  = float(np.std(arr))

    for i in range(N_MFCC):
        add_stats(f"mfcc{i+1}", mfcc[i])
        add_stats(f"dmfcc{i+1}", dmfcc[i])

    add_stats("rms", rms)
    add_stats("zcr", zcr)
    add_stats("centroid", centroid)
    add_stats("bandwidth", bandwidth)

    # silence features (measure silence, don't remove)
    thr = np.percentile(rms, 20)
    silent = rms < thr
    feats["silence_ratio"] = float(np.mean(silent))
    feats["duration_sec"] = float(len(y) / SR)

    return feats

def list_audio_files(folder):
    exts = ("*.wav", "*.mp3", "*.flac", "*.m4a", "*.ogg")
    files = []
    for e in exts:
        files += glob.glob(os.path.join(folder, e))
    return sorted(files)

rows = []
for language in ["italian", "english"]:
    for label in ["PD", "HC"]:
        folder = os.path.join(DATA_ROOT, language, label)
        files = list_audio_files(folder)
        print(language, label, "files:", len(files))

        for fp in files:
            feats = extract_features(fp)
            if feats is None:
                continue
            feats["language"] = language
            feats["label"] = label
            feats["file_path"] = fp
            rows.append(feats)

df = pd.DataFrame(rows)
df.to_csv("/content/features.csv", index=False)

print("Saved features.csv with shape:", df.shape)
df.head()

italian PD files: 0
italian HC files: 0
english PD files: 16
english HC files: 21
Saved features.csv with shape: (37, 65)


,mfcc1_mean,mfcc1_std,dmfcc1_mean,dmfcc1_std,mfcc2_mean,mfcc2_std,dmfcc2_mean,dmfcc2_std,mfcc3_mean,mfcc3_std,...,zcr_std,centroid_mean,centroid_std,bandwidth_mean,bandwidth_std,silence_ratio,duration_sec,language,label,file_path
0,-440.424347,104.324921,0.002255,14.012710,74.367577,48.744385,0.003054,10.191586,41.430790,27.914551,...,0.126786,1848.286273,1098.293389,1905.074444,417.145008,0.200041,157.556688,english,PD,/content/drive/MyDrive/Colab Notebooks/Thesis/...
1,-453.525604,119.625954,0.016781,14.886793,100.566765,43.602791,0.003594,8.646607,18.105577,29.934574,...,0.082682,1508.487578,776.872213,1713.723791,430.656107,0.200104,122.482313,english,PD,/content/drive/MyDrive/Colab Notebooks/Thesis/...
2,-458.686951,120.616051,-0.034241,14.709138,89.630020,45.382824,0.004831,9.156718,29.887951,26.993450,...,0.090793,1576.640145,852.734004,1739.969652,394.762097,0.200107,178.920125,english,PD,/content/drive/MyDrive/Colab Notebooks/Thesis/...
3,-531.569153,102.998207,-0.007342,12.860456,92.156578,49.145237,0.002987,10.290421,43.460304,29.936522,...,0.125781,1646.870589,1127.612581,1769.191084,457.282698,0.200000,147.665750,english,PD,/content/drive/MyDrive/Colab Notebooks/Thesis/...
4,-462.067566,60.863453,0.038071,8.636037,59.070648,40.787121,0.025607,6.150354,36.049557,18.865362,...,0.048331,1034.366202,828.471047,1585.203490,635.143504,0.200137,93.361437,english,PD,/content/drive/MyDrive/Colab Notebooks/Thesis/...


In [ ]:
df